# 01 - Carga y Exploración Inicial - OULAD

**Objetivo específico 1:** Identificar las variables académicas, conductuales y temporales
disponibles en el dataset, mediante técnicas de preprocesamiento y análisis exploratorio,
con el fin de establecer su pertinencia para el estudio del abandono en un entorno LMS.

Proyecto: Modelo predictivo del abandono estudiantil en entornos LMS (OULAD)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga de las 7 tablas

In [ ]:
tablas = {
    'courses': RAW_DIR / 'courses.csv',
    'assessments': RAW_DIR / 'assessments.csv',
    'vle': RAW_DIR / 'vle.csv',
    'studentInfo': RAW_DIR / 'studentInfo.csv',
    'studentRegistration': RAW_DIR / 'studentRegistration.csv',
    'studentAssessment': RAW_DIR / 'studentAssessment.csv',
    'studentVle': RAW_DIR / 'studentVle.csv',  # tabla más pesada (~10M filas)
}

dfs = {}
for nombre, ruta in tablas.items():
    dfs[nombre] = pd.read_csv(ruta)
    print(f"{nombre:22s} -> shape: {dfs[nombre].shape}")

## 2. Vista general por tabla (dtypes + nulos)

In [ ]:
def resumen_tabla(df, nombre):
    print(f"\n{'='*60}\n{nombre}\n{'='*60}")
    resumen = pd.DataFrame({
        'dtype': df.dtypes,
        'n_nulos': df.isnull().sum(),
        '%_nulos': (df.isnull().sum() / len(df) * 100).round(2),
        'n_unicos': df.nunique()
    })
    print(resumen)
    return resumen

for nombre, df in dfs.items():
    resumen_tabla(df, nombre)

## 3. Variable objetivo: `studentRegistration`

El abandono se define (según anteproyecto, sección 8.1.3, A3.1) como la presencia de un
valor no nulo en `date_unregistration`.

In [ ]:
reg = dfs['studentRegistration']
reg.head()

In [ ]:
reg['abandono'] = reg['date_unregistration'].notnull().astype(int)

n_total = len(reg)
n_abandono = reg['abandono'].sum()
pct_abandono = n_abandono / n_total * 100

print(f"Total registros (estudiante x módulo x presentación): {n_total}")
print(f"Registros con abandono (retiro voluntario): {n_abandono} ({pct_abandono:.2f}%)")
print(f"Registros sin abandono: {n_total - n_abandono} ({100 - pct_abandono:.2f}%)")

In [ ]:
reg['abandono'].value_counts().plot(kind='bar', title='Distribución de la variable objetivo (abandono)')

## 4. Verificación de llaves de unión entre tablas

Las llaves principales para relacionar las tablas son:
- `code_module` + `code_presentation` (identifica un curso-presentación)
- `id_student` (identifica al estudiante)


In [ ]:
# Verificar que id_student en studentInfo coincide con studentRegistration
ids_info = set(dfs['studentInfo']['id_student'].unique())
ids_reg = set(dfs['studentRegistration']['id_student'].unique())

print(f"Estudiantes únicos en studentInfo: {len(ids_info)}")
print(f"Estudiantes únicos en studentRegistration: {len(ids_reg)}")
print(f"Diferencia (en info pero no en reg): {len(ids_info - ids_reg)}")
print(f"Diferencia (en reg pero no en info): {len(ids_reg - ids_info)}")

In [ ]:
# Verificar llave compuesta code_module + code_presentation en courses vs studentInfo
cursos = set(dfs['courses'].apply(lambda r: (r['code_module'], r['code_presentation']), axis=1))
cursos_info = set(dfs['studentInfo'].apply(lambda r: (r['code_module'], r['code_presentation']), axis=1))

print(f"Módulo-presentaciones en courses: {len(cursos)}")
print(f"Módulo-presentaciones en studentInfo: {len(cursos_info)}")
print(f"Coinciden: {cursos == cursos_info}")

## 5. Vista preliminar de `studentVle` (interacción)

Esta es la tabla más pesada y la base para las variables conductuales/temporales
del Objetivo 2.

In [ ]:
svle = dfs['studentVle']
print(f"Shape: {svle.shape}")
print(f"Rango de 'date' (días relativos al inicio del curso): {svle['date'].min()} a {svle['date'].max()}")
svle.head()

## 6. Próximos pasos

- Guardar hallazgos de esta exploración (variables candidatas, calidad de datos)
- Pasar al notebook `02_feature_engineering.ipynb` (Objetivo 2): construcción de
  variables de interacción, continuidad, inactividad y desempeño académico inicial.
